# VLM QLoRA Training — Kaggle Session

## Before running — Kaggle Dataset setup (one-time)

Create **two private Kaggle Datasets** under your account:

| Dataset slug | Contents | When |
|---|---|---|
| `vlm-projector` | `projector_stage1.pt` (922 MB) | Upload once |
| `vlm-session-state` | checkpoint folder + `cls_head.pt` + `faiss_index/` + `logs/stage2.jsonl` | Update after each session |

**Session 1:** Only attach `vlm-projector`. Set `IS_FIRST_SESSION = True`.

**Session 2+:** Attach both datasets. Set `IS_FIRST_SESSION = False`.

**Kaggle Secrets** (Settings → Secrets — Add Secret):
- `HF_TOKEN`
- `SEMANTIC_SCHOLAR_API_KEY`


In [ ]:
# ── USER CONFIG — edit these before each session ─────────────────────────────

IS_FIRST_SESSION  = True   # True = session 1 (builds FAISS, no prior checkpoint)
                            # False = session 2+ (restores FAISS + checkpoint)

# ── 4 GB VRAM HYPOTHESIS TEST ─────────────────────────────────────────────────
# True  → hard-caps PyTorch to 4 GB on the T4, simulating an RTX 3050 4 GB.
#         Use this to verify the model trains within the constraint before
#         committing to a full multi-session run.
# False → uses full 16 GB T4 for fast production training (~35–50 s/step).
SIMULATE_4GB_VRAM = True

# Kaggle Dataset slugs — must match what you created in your Kaggle account
PROJECTOR_DATASET = "vlm-projector"       # contains projector_stage1.pt
STATE_DATASET     = "vlm-session-state"   # contains checkpoint + faiss_index

# Your Kaggle username (needed to build input paths)
KAGGLE_USERNAME   = "your-kaggle-username"  # e.g. "sujalprasad"

# Training config
MAX_PAIRS  = 4000
EPOCHS     = 3
# grad_accum=8 matches the laptop exactly when simulating 4 GB.
# grad_accum=4 halves accumulation steps for faster full-speed runs.
GRAD_ACCUM = 8 if SIMULATE_4GB_VRAM else 4
SAVE_EVERY = 250
LR         = 2e-4

print("Config loaded.")
print(f"  Session type   : {'FIRST (builds FAISS)' if IS_FIRST_SESSION else 'RESUME'}")
print(f"  4GB simulation : {'YES — capped at 4 GB, grad_accum=8' if SIMULATE_4GB_VRAM else 'NO — full 16 GB, grad_accum=4'}")
print(f"  Grad accum     : {GRAD_ACCUM}")


In [ ]:
# ── Install packages ──────────────────────────────────────────────────────────
# Kaggle already has torch, transformers, datasets, numpy, Pillow
# We only need the extras

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("peft>=0.19.1")
pip("bitsandbytes>=0.49.2")
pip("accelerate>=1.13.0")
pip("faiss-cpu==1.13.2")
pip("sentence-transformers")
pip("python-dotenv")

print("Packages ready.")


In [ ]:
# ── Secrets & environment variables ──────────────────────────────────────────

import os, torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"]                 = secrets.get_secret("HF_TOKEN")
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = secrets.get_secret("SEMANTIC_SCHOLAR_API_KEY")
os.environ["CUBLAS_WORKSPACE_CONFIG"]  = ":4096:8"

# HuggingFace login (needed for MIMIC credentialed access)
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

# ── 4 GB cap (must be set BEFORE any CUDA allocation) ─────────────────────────
if SIMULATE_4GB_VRAM:
    # T4 = 16 GB → fraction 0.25 = 4 GB hard ceiling.
    # We use 3.8/16 not 4/16 to account for ~200 MB Windows WDDM overhead
    # that exists on the real laptop but not on Kaggle Linux — keeps the
    # simulation honest.
    torch.cuda.set_per_process_memory_fraction(3.8 / 16, 0)
    os.environ["MEDDIAG_MAX_VRAM_GB"] = "3.8"
    print("4 GB VRAM simulation ACTIVE")
    print(f"  Fraction set : {3.8/16:.4f}  ({3.8:.1f} GB / 16 GB T4)")
    print(f"  Any allocation beyond 3.8 GB will OOM exactly like an RTX 3050")
else:
    os.environ["MEDDIAG_MAX_VRAM_GB"] = "14"
    print("Full 16 GB mode — no VRAM cap")

print("Secrets loaded, HF login OK.")


In [ ]:
# ── Clone repo & set working directory ───────────────────────────────────────

import subprocess, os

REPO_URL  = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_DIR  = "/kaggle/working/vlm"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, REPO_DIR])
    print(f"Repo cloned → {REPO_DIR}")
else:
    # Discard local patches to run_pipeline.sh before pulling (cell 7 re-applies them)
    subprocess.call(["git", "-C", REPO_DIR, "checkout", "run_pipeline.sh"])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print(f"Repo updated → {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ── Copy assets from Kaggle input datasets into working tree ─────────────────

import shutil, os
from pathlib import Path

PROJECTOR_SRC = Path(f"/kaggle/input/{PROJECTOR_DATASET}/projector_stage1.pt")
MODELS_DIR    = Path(REPO_DIR) / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Always copy projector
shutil.copy2(PROJECTOR_SRC, MODELS_DIR / "projector_stage1.pt")
print(f"Projector copied  ({PROJECTOR_SRC.stat().st_size / 1e6:.0f} MB)")

if not IS_FIRST_SESSION:
    STATE_SRC = Path(f"/kaggle/input/{STATE_DATASET}")

    # Restore FAISS index
    faiss_src = STATE_SRC / "faiss_index"
    faiss_dst = Path(REPO_DIR) / "faiss_index"
    if faiss_src.exists():
        if faiss_dst.exists():
            shutil.rmtree(faiss_dst)
        shutil.copytree(faiss_src, faiss_dst)
        print(f"FAISS index restored → {faiss_dst}")

    # Restore latest LoRA checkpoint(s)
    import re
    ckpt_pattern = re.compile(r"lora_step(\d+)$")
    for item in STATE_SRC.iterdir():
        if ckpt_pattern.match(item.name) and item.is_dir():
            dst = MODELS_DIR / item.name
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(item, dst)
            print(f"Checkpoint restored  → {dst}")

    # Restore ClassificationHead
    cls_src = STATE_SRC / "cls_head.pt"
    if cls_src.exists():
        shutil.copy2(cls_src, MODELS_DIR / "cls_head.pt")
        print("ClassificationHead   restored")

    # Restore training log (for continuity)
    logs_dir = Path(REPO_DIR) / "logs"
    logs_dir.mkdir(exist_ok=True)
    log_src = STATE_SRC / "stage2.jsonl"
    if log_src.exists():
        shutil.copy2(log_src, logs_dir / "stage2.jsonl")
        print("Training log         restored")

print("\nAssets ready.")


In [ ]:
# ── Fix pipeline state for Kaggle ─────────────────────────────────────────────
# The repo's .pipeline_state already has steps 0,1,2 marked done.
# On session 1 we need step 1 (FAISS build) to run since we have no index yet.

from pathlib import Path

state_path = Path(REPO_DIR) / "logs" / ".pipeline_state"
state_path.parent.mkdir(exist_ok=True)

if IS_FIRST_SESSION:
    # Write state with only step0 and step2 done; step1 (FAISS) must rebuild
    state_path.write_text("\nstep0\nstep0\nstep0\nstep0\nstep0\nstep0\nstep0\nstep2\n")
    print("Pipeline state: FAISS will be rebuilt (session 1)")
else:
    # Keep the repo's state as-is (step1 already done — FAISS was restored above)
    print(f"Pipeline state: using repo defaults (FAISS + stage 1 already done)")

# Remove stale lock if any
lock = Path(REPO_DIR) / "logs" / ".pipeline.lock"
lock.unlink(missing_ok=True)
print("Lock cleared.")


In [ ]:
# ── Patch run_pipeline.sh with Kaggle-optimised config ───────────────────────
# Overwrites GRAD_ACCUM and MAX_PAIRS inline; no permanent file change needed.

pipeline_sh = Path(REPO_DIR) / "run_pipeline.sh"
text = pipeline_sh.read_text()

import re
text = re.sub(r"(GRAD_ACCUM=)\d+",   f"GRAD_ACCUM={GRAD_ACCUM}",   text)
text = re.sub(r"(MAX_PAIRS_S2=)\d+", f"MAX_PAIRS_S2={MAX_PAIRS}",  text)

pipeline_sh.write_text(text)
print(f"run_pipeline.sh patched: GRAD_ACCUM={GRAD_ACCUM}, MAX_PAIRS_S2={MAX_PAIRS}")


In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
# Streams output live. Runs until Kaggle's 12-hour session limit kills it
# or until training completes. Checkpoints save every 250 steps.

import subprocess, os

env = os.environ.copy()  # already has HF_TOKEN, MEDDIAG_MAX_VRAM_GB, CUBLAS config

cmd = ["bash", "run_pipeline.sh", "--resume"]

mode = "4 GB simulation" if SIMULATE_4GB_VRAM else "full 16 GB"
print(f"Starting pipeline in {mode} mode...")
print("=" * 60)

proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\n[notebook] Training interrupted — checkpoint already saved at last step.")

proc.wait()
print(f"\n[notebook] Process exited with code {proc.returncode}")

if SIMULATE_4GB_VRAM and proc.returncode == 0:
    print("\n✓ Hypothesis CONFIRMED: model trains within 4 GB VRAM.")
elif SIMULATE_4GB_VRAM and proc.returncode != 0:
    print("\n✗ Check logs above — OOM may have occurred beyond the 3.8 GB cap.")


In [ ]:
# ── Save session state for next session ───────────────────────────────────────
# Run this cell BEFORE the session ends (or Kaggle will save /kaggle/working
# automatically when you click "Save Version").
#
# Everything copied to /kaggle/working/session_state/ becomes the output
# dataset you attach as "vlm-session-state" in the next session.

import shutil, re
from pathlib import Path

OUT_DIR   = Path("/kaggle/working/session_state")
OUT_DIR.mkdir(exist_ok=True)
MODELS    = Path(REPO_DIR) / "models"
LOGS      = Path(REPO_DIR) / "logs"

# Keep only the 2 latest lora checkpoints (pruning already runs during training,
# but we double-check here before uploading)
ckpt_pattern = re.compile(r"lora_step(\d+)$")
ckpts = sorted(
    [(int(m.group(1)), p) for p in MODELS.iterdir() if (m := ckpt_pattern.match(p.name))]
)
for _, ckpt_path in ckpts[-2:]:   # copy 2 most recent
    dst = OUT_DIR / ckpt_path.name
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(ckpt_path, dst)
    print(f"Saved checkpoint  → {dst.name}")

# ClassificationHead
cls = MODELS / "cls_head.pt"
if cls.exists():
    shutil.copy2(cls, OUT_DIR / "cls_head.pt")
    print("Saved cls_head.pt")

# FAISS index (only needed once; reuse across all sessions)
faiss_src = Path(REPO_DIR) / "faiss_index"
if faiss_src.exists():
    faiss_dst = OUT_DIR / "faiss_index"
    if faiss_dst.exists(): shutil.rmtree(faiss_dst)
    shutil.copytree(faiss_src, faiss_dst)
    print("Saved faiss_index/")

# Training log
log = LOGS / "stage2.jsonl"
if log.exists():
    shutil.copy2(log, OUT_DIR / "stage2.jsonl")
    print("Saved stage2.jsonl")

# Show total size
total = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())
print(f"\nTotal session state: {total / 1e6:.0f} MB → /kaggle/working/session_state/")
print("Upload this folder as 'vlm-session-state' dataset for your next session.")


## Between-session checklist

1. After training stops, run the **Save session state** cell above.
2. Go to **Output** tab → download `session_state/` folder.
3. Update your `vlm-session-state` Kaggle Dataset with the new contents.
4. Next session: set `IS_FIRST_SESSION = False`, attach both datasets, run all cells.

## Modes at a glance

| | 4 GB simulation (`SIMULATE_4GB_VRAM=True`) | Full speed (`SIMULATE_4GB_VRAM=False`) |
|---|---|---|
| Purpose | Verify hypothesis before committing | Production multi-session training |
| VRAM cap | 3.8 GB (matches RTX 3050 + WDDM) | 14 GB |
| `grad_accum` | 8 (identical to laptop) | 4 |
| Speed | ~100 s/step (same as laptop) | ~35–50 s/step |
| Passes = hypothesis confirmed | vram log stays ≤ 3.8 GB, no OOM | — |

## Recommended workflow

1. **First**: run with `SIMULATE_4GB_VRAM=True` for ~50–100 steps. Confirm vram stays ≤ 3.8 GB.
2. **Then**: flip to `False`, restart from checkpoint, finish training at full speed across sessions.
